# 🃏 Part 3: Capstone Project - Object-Oriented Blackjack

* **Source:** [freeCodeCamp Python for Beginners](https://www.youtube.com/watch?v=eWRfhZUzrAc)
* **Objective**: Build a fully playable, interactive terminal Blackjack game to synthesize core Python concepts, acting as a final test of everything learned in Parts 1 and 2.
* **Core Concepts**: Object-Oriented Programming (Classes, Instances, Methods), Loops (`while`/`for`), Control Flow, and State Management.
* **Game Architecture**:
  1. **Assets**: Global constants defining the deck properties (Suits, Ranks, Values).
  2. **Core Entities**: `Card` and `Deck` classes to handle creating and shuffling the physical cards.
  3. **Player State**: A `Hand` class to manage drawn cards and dynamically calculate scores (including the tricky 1 or 11 Ace logic).
  4. **The Engine**: The main game loop that handles user inputs (Hit/Stand) and evaluates win/loss conditions.

### 1. The Assets (Card and Deck Classes)
* **`Card`**: A simple data container representing a single playing card. We override the `__str__` dunder method so it prints cleanly (e.g., "A of spades").
* **`Deck`**: Generates a standard 52-card deck using nested `for` loops. It handles its own shuffling and dealing logic.

In [1]:
import random

class Card:
    """Represents a single playing card."""
    def __init__(self, suit, rank):
        self.suit = suit
        self.rank = rank
        
    def __str__(self): 
        # Overrides the default print output to be human-readable
        return f"{self.rank['rank']} of {self.suit}"
        
        
class Deck:
    """Generates and manages a 52-card deck."""
    def __init__(self):
        self.cards = []
        suits = ["hearts", "diamonds", "clubs", "spades"]
        
        # A list of dictionaries mapping the string rank to its integer value
        ranks = [
                {"rank": "A", "value": 11},
                {"rank": "2", "value": 2},
                {"rank": "3", "value": 3},
                {"rank": "4", "value": 4},
                {"rank": "5", "value": 5},
                {"rank": "6", "value": 6},
                {"rank": "7", "value": 7},
                {"rank": "8", "value": 8},
                {"rank": "9", "value": 9},
                {"rank": "10", "value": 10},
                {"rank": "J", "value": 10},
                {"rank": "Q", "value": 10},
                {"rank": "K", "value": 10}
        ]

        # Nested loops to generate every combination of suit and rank
        for suit in suits:
            for rank in ranks:
                self.cards.append(Card(suit, rank))

    def shuffle(self):
        """Randomizes the order of the cards in the deck."""
        if len(self.cards) > 1:
            random.shuffle(self.cards)

    def deal(self, number):
        """Removes a specified number of cards from the deck and returns them."""
        cards_dealt = []
        for x in range(number):
            if len(self.cards) > 0:
                card = self.cards.pop()
                cards_dealt.append(card)
        return cards_dealt

### 2. Player State (The Hand Class)
* **`Hand`**: Manages the cards currently held by the player or the dealer. 
* **Dynamic Ace Logic**: The `calculate_value` method dynamically checks if the hand has busted (> 21) and if it contains an Ace. If both are true, it subtracts 10, effectively turning the Ace from an 11 into a 1.

In [2]:
class Hand:
    """Manages the cards drawn by a player or dealer and calculates the score."""
    def __init__(self, dealer=False):
        self.cards = []
        self.value = 0
        self.dealer = dealer
        
    def add_card(self, card_list):
        """Extends the current hand with new cards."""
        self.cards.extend(card_list)
        
    def calculate_value(self):
        """Calculates the total hand value, dynamically adjusting Aces if busting."""
        self.value = 0
        has_ace = False
        
        for card in self.cards:
            card_value = int(card.rank["value"])
            self.value += card_value
            if card.rank["rank"] == "A":
                has_ace = True

        # If the hand busts but has an Ace, convert the 11 to a 1
        if has_ace and self.value > 21:
            self.value -= 10
            
    def get_value(self):
        self.calculate_value()
        return self.value
        
    def is_blackjack(self):
        return self.get_value() == 21
    
    def display(self, show_all_dealer_cards=False):
        """Prints the hand to the terminal. Hides the dealer's first card by default."""
        print(f'''{"Dealer's" if self.dealer else "Your"} hand:''')
        
        for index, card in enumerate(self.cards):
            # Hide the dealer's first card unless the round is over or they have blackjack
            if index == 0 and self.dealer and not show_all_dealer_cards and not self.is_blackjack():
                print("Hidden")
            else:
                print(card)
            
        if not self.dealer:
            print("Value:", self.get_value())
        print()

### 3. The Engine (The Game Class)
* **`Game`**: The main controller. It handles user input loops, object instantiation (deck and hands) for each new round, and delegates the win/loss evaluation to a helper method.

In [3]:
class Game:
    """The main game loop and logic controller."""
    def play(self):
        game_number = 0
        games_to_play = 0
        
        # 1. Setup Loop: Determine how many games to play
        while games_to_play <= 0:
            try:
                games_to_play = int(input("How many games do you want to play? "))
            except ValueError:
                print("You must enter a number.")
                
        # 2. Main Game Loop
        while game_number < games_to_play:
            game_number += 1
            
            # Initialize a fresh, shuffled deck and empty hands for the round
            deck = Deck()
            deck.shuffle()
                
            player_hand = Hand()
            dealer_hand = Hand(dealer=True)
            
            # Initial Deal: 2 cards each
            for i in range(2):
                player_hand.add_card(deck.deal(1))
                dealer_hand.add_card(deck.deal(1))
                
            print()
            print("*" * 30)
            print(f"Game {game_number} of {games_to_play}")
            print("*" * 30)
            player_hand.display()
            dealer_hand.display()
            
            # Check for instant Blackjacks before hitting
            if self.check_winner(player_hand, dealer_hand):
                continue
            
            # 3. Player Phase (Hit or Stand)
            choice = ""
            while player_hand.get_value() < 21 and choice not in ["s", "stand"]:
                choice = input("Please choose 'Hit' or 'Stand': ").lower()
                print()
                
                # Input validation
                while choice not in ["h", "s", "hit", "stand"]:
                    choice = input("Please enter 'Hit' or 'Stand' (or H/S): ").lower()
                    print()
                    
                if choice in ["hit", "h"]:
                    player_hand.add_card(deck.deal(1))
                    player_hand.display()
                    
            # Check if player busted after hitting
            if self.check_winner(player_hand, dealer_hand):
                continue
            
            player_hand_value = player_hand.get_value()
            dealer_hand_value = dealer_hand.get_value()
            
            # 4. Dealer Phase (Must hit until 17)
            while dealer_hand_value < 17:
                dealer_hand.add_card(deck.deal(1))
                dealer_hand_value = dealer_hand.get_value()
            
            # Reveal full dealer hand
            dealer_hand.display(show_all_dealer_cards=True)
            
            if self.check_winner(player_hand, dealer_hand):
                continue
            
            # 5. Final Evaluation Phase
            print("Final Results:")
            print("Your hand:", player_hand_value)
            print("Dealer's hand:", dealer_hand_value)
            
            self.check_winner(player_hand, dealer_hand, game_over=True)
            
        print("\nThanks for playing!")
                        
    def check_winner(self, player_hand, dealer_hand, game_over=False):
        """Helper method to determine win, loss, tie, or bust conditions."""
        if not game_over:
            # Check conditions during the active play phase
            if player_hand.get_value() > 21:
                print("You busted! Dealer wins.")
                return True
            elif dealer_hand.get_value() > 21:
                print("Dealer busted! You win.")
                return True
            elif dealer_hand.is_blackjack() and player_hand.is_blackjack():
                print("Both players have blackjack! It's a tie.")
                return True
            elif player_hand.is_blackjack():
                print("You have blackjack! You win.")
                return True 
            elif dealer_hand.is_blackjack():
                print("Dealer has blackjack! Dealer wins.")
                return True
        else:
            # Check final scores after both phases are complete
            if player_hand.get_value() > dealer_hand.get_value():
                print("You win!")
            elif player_hand.get_value() == dealer_hand.get_value(): 
                print("It's a tie!")
            else:
                print("Dealer wins!")
        
        # Returns False if the game should continue to the next phase
        return False

# --- Execute the Game ---
g = Game()
g.play()


******************************
Game 1 of 3
******************************
Your hand:
Q of clubs
K of diamonds
Value: 20

Dealer's hand:
Hidden
9 of diamonds


Dealer's hand:
7 of spades
9 of diamonds
6 of hearts

Dealer busted! You win.

******************************
Game 2 of 3
******************************
Your hand:
5 of spades
10 of clubs
Value: 15

Dealer's hand:
Hidden
9 of clubs


Your hand:
5 of spades
10 of clubs
10 of diamonds
Value: 25

You busted! Dealer wins.

******************************
Game 3 of 3
******************************
Your hand:
3 of hearts
8 of diamonds
Value: 11

Dealer's hand:
Hidden
9 of clubs


Your hand:
3 of hearts
8 of diamonds
8 of spades
Value: 19


Dealer's hand:
10 of spades
9 of clubs

Final Results:
Your hand: 19
Dealer's hand: 19
It's a tie!

Thanks for playing!
